In [510]:
import numpy as np

# Quantum interpreter

An implementation of a quantum interpreter based on Robert Smith's [tutorial](https://www.stylewarning.com/posts/quantum-interpreter/).

## Quantum circuits

The fundamental operations of a quantum computer are gates and measurements. Gates are unitary matrices that modify the quantum state. Measurements observe the current quantum state and collapse it into a classical state.

You can think of gates and measurements are basic instructions in a quantum computing assembly language. Another interpretation sees them as basic circuit elements in a quantum circuit similar to logic gates NOT, AND, OR in digital circuits.

In [511]:
type Matrix = list[list[np.complex64]]

class Operator:
    """A quantum operator."""

    def __init__(self, name: str, matrix: Matrix):
        """Create a quantum operator from a unitary matrix.
        
        Arguments:
           gate         The gate matrix.
           name         An optional name for the operator.

        Asserts the gate matrix is unitary.
        """
        self.name = name
        self.matrix = np.array(matrix)
        self.numberOfQubits = int(np.log2(self._numberOfStates()))
        self._assertUnitary()

    def __repr__(self):
        return f"Operator {self.name}:\n{self.matrix}"

    def _assertUnitary(self):
        """Asserts the operator is a unitary matrix."""
        u = self.matrix
        i = np.identity(self._numberOfStates())
        assert(np.allclose(u @ u.conjugate().transpose(), i, atol=0.001))

    def _numberOfStates(self) -> int:
        """Asserts the operator is a unitary matrix."""
        return self.matrix.shape[0]

In [512]:
# Hadamard gate that puts a single qubit into superposition.
SQRT_ONE_HALF = np.sqrt(0.5, dtype=np.complex64)
H = Operator("H", [
    [SQRT_ONE_HALF, SQRT_ONE_HALF],     # |0> -> |+>
    [SQRT_ONE_HALF, -SQRT_ONE_HALF],    # |1> -> |->
])

# Controlled NOT on two qubits. The least qubit is the control qubit.
# If the control qubit is (0), the target is not modified.
# If the control qubit is (1), the target is inverted.
CNOT = Operator("CNOT", [
    [1, 0, 0, 0],   # |00> -> |00>
    [0, 1, 0, 0],   # |01> -> |01>
    [0, 0, 0, 1],   # |10> -> |11>
    [0, 0, 1, 0],   # |11> -> |10>
])

In [513]:
type QubitList = list[int]

class Gate:
    """A quantum gate."""

    def __init__(self, u: Operator, qubits : QubitList):
        """Create a quantum gate.
        
        Arguments:
            u           A unitary matrix describing the gate.
            qubits      The indices of the qubits in the quantum
                        state to which the gate applies.
        """
        self.operator = u
        self.qubits = qubits

    def __repr__(self):
        op = self.operator
        return f"Gate {op.name} on qubits {self.qubits}:\n{op.matrix}"

class Measure:
    """A quantum measurement."""

    def __init__(self):
        """Create a quantum measurement."""

    def __repr__(self):
        return f"Measure"

In [514]:
ExampleBell25 = [
    Gate(H, [2]),
    Gate(CNOT, [2, 5]),
    Measure(),
]
ExampleBell25

[Gate H on qubits [2]:
 [[ 0.70710677+0.j  0.70710677+0.j]
  [ 0.70710677+0.j -0.70710677-0.j]],
 Gate CNOT on qubits [2, 5]:
 [[1 0 0 0]
  [0 1 0 0]
  [0 0 0 1]
  [0 0 1 0]],
 Measure]

In [515]:
class Circuit:
    """A quantum circuit."""

    def __init__(self, n: int):
        """Create an empty quantum circuit of n qubits.
        
        Arguments:
            n       The number of qubits in the circuit.
        """
        self.numberOfQubits = n
        self.operations = []

    def add(self, gate: Operator, qubits : QubitList):
        """Add a gate to the circuit."""
        assert(all(qubit < self.numberOfQubits for qubit in qubits))
        self.operations.append(Gate(gate, qubits))

    def measure(self):
        """Measure and collapse the current quantum state."""
        self.operations.append(Measure())

In [516]:
ExampleBell25 = Circuit(6)
ExampleBell25.add(H, [2])
ExampleBell25.add(CNOT, [2, 5])
ExampleBell25.measure()
ExampleBell25.operations

[Gate H on qubits [2]:
 [[ 0.70710677+0.j  0.70710677+0.j]
  [ 0.70710677+0.j -0.70710677-0.j]],
 Gate CNOT on qubits [2, 5]:
 [[1 0 0 0]
  [0 1 0 0]
  [0 0 0 1]
  [0 0 1 0]],
 Measure]

## Quantum states

A single qubit has two classical states: $|0\rangle$ and $|1\rangle$. Two qubits have four classical states: $|00\rangle$, $|01\rangle$, $|10\rangle$ and $|11\rangle$, and so on. The rightmost qubit in $|x..x\rangle$ is the least significant qubit as with ordinary binary numbers.

Unlike classic bits, qubits can be in a superposition ofs classical states, that is, the state of a qubit is a linear combination ("weighted average") of the classical states. For example, for two qubits the linear combination might be:

| A | B | AB | Amplitude | Probability |
|-|-|-|-|-|
| $\|0\rangle$ | $\|0\rangle$ | $\|00\rangle$ | $1/\sqrt(2)$ | 50% |
| $\|0\rangle$ | $\|1\rangle$ | $\|01\rangle$ | 0 | 0% |
| $\|1\rangle$ | $\|0\rangle$ | $\|10\rangle$ | 0 | 0% |
| $\|1\rangle$ | $\|1\rangle$ | $\|11\rangle$ | $1/\sqrt(2)$ | 50% |

The "weights" in the "weighted average" are the complex probability amplitudes of the corresponding classical state. The squares of the probability amplitudes have to sum to 1.

A quantum state consists of the probability amplitudes of the classical states.
The first entry in the quantum state vector corresponds to $|...000\rangle$ and
the last entry corresonds to $|...111\rangle$. We can use the binary number that
corresponds a classical state to index into the quantum state vector and to find
its probability amplitude.

Since the quantum state is a vector, transforming the state or applying an operation to it is the same as multiplying a matrix with the quantum state vector.

A measurement on the quantum state vector collapses the quantum state into a classical state. We view the quantum state as a discrete probability distribution and we sample from it.

In [517]:
type Counts = np.ndarray[tuple[int], np.dtype[np.int64]]
type DiscreteDistribution = np.ndarray[tuple[int], np.dtype[np.float64]]

class State:
    """A quantum state."""

    def __init__(self, n: int):
        """Create a quantum state for n qubits."""
        self.numberOfQubits = n
        self.collapse(0)

    def apply(self, operator: Operator):
        """Apply a quantum operator to the quantum state."""
        # A quantum operator is a unitary matrix. Applying the
        # operator is the same as matrix-vector multiplication.
        self.state = operator.matrix.dot(self.state)

    def measure(self, n: int = 1):
        """Measure and collapse the quantum state.
        
        Arguments:
            n       The number of times to sample from the
                    probability distribution of the quantum state.
        """
        counts = self.sample(n)
        self.collapse(np.argmax(counts))

    def sample(self, n: int = 1) -> Counts:
        """Sample n times from the probability distribution of the quantum
        state.
        
        Arguments:
            n       The number of times to sample from the
                    probability distribution of the quantum state.
        """
        probabilities = self.probabilities()
        states = list(range(len(probabilities)))

        samples = np.random.choice(states, n, p=probabilities)
        return np.unique_counts(samples).counts

    def probabilities(self) -> DiscreteDistribution:
        """Return the discrete probability distribution corresponding
        to the quantum state."""
        return np.abs(np.square(self.state))

    def collapse(self, i : int):
        """Collapse the quantum state to the classical state i.
        
        Arguments:
            i       The index of the classical state.
        """
        self.state = np.zeros(shape=2**self.numberOfQubits, dtype=np.complex64)
        self.state[i] = 1+0j

    def __repr__(self):
        n = self.numberOfQubits
        result = f"State of {n} qubits with amplitudes and probabilities:\n"
        for i, entry in enumerate(self.state):
            result += f"|{i:0{n}b}>{entry:20.6}{abs(entry**2):20.6}\n"
        return result

In [518]:
State(1)

State of 1 qubits with amplitudes and probabilities:
|0>              (1+0j)                 1.0
|1>                  0j                 0.0

In [519]:
State(2)

State of 2 qubits with amplitudes and probabilities:
|00>              (1+0j)                 1.0
|01>                  0j                 0.0
|10>                  0j                 0.0
|11>                  0j                 0.0

In [520]:
s = State(1)
s.apply(H)
s

State of 1 qubits with amplitudes and probabilities:
|0>       (0.707107+0j)                 0.5
|1>       (0.707107+0j)                 0.5

In [521]:
s.sample(100)

array([45, 55])

In [522]:
s.measure(100)
s

State of 1 qubits with amplitudes and probabilities:
|0>              (1+0j)                 1.0
|1>                  0j                 0.0

## Gates

We represent the $2^n$ classical states for $n$ qubits as a $2^n$-dimensional vector. Therefore a quantum gate operating on $n$ qubits is a $2^n$-dimensional unitary matrix.

Simple quantum gates are I (Identity) and X (NOT).

In [523]:
# Identity.
I = Operator("I", [
    [1, 0],     # |0> -> |0>
    [0, 1],     # |1> -> |1>
])

# X gate (NOT gate).
X = Operator("X", [
    [0, 1],     # |0> -> |1>
    [1, 0],     # |1> -> |0>
])

### Gates on single qubits

A gate on a single qubit in a quantum state modifies only that qubit. That is, we apply the gate $g$ to qubit $i$ and we apply the identity $I$ to the other qubits:

\begin{equation}
\operatorname{lift}(g, i, n) :=
\underbrace{\mathsf{I} \otimes \mathsf{I} \otimes \cdots}_{n-i-1\text{ factors}}
\otimes g \otimes
\underbrace{\cdots \otimes \mathsf{I}}_{i\text{ factors}},
\end{equation}

([Equation (2) in the tutorial](https://www.stylewarning.com/posts/quantum-interpreter/))

And we can generalize that if the gate applies to $k$ adjacent qubits:

\begin{equation}
\operatorname{lift}(g, i, n) := \underbrace{\mathsf{I} \otimes \mathsf{I} \otimes \cdots}_{n-i-k\text{ factors}}
\otimes g \otimes
\underbrace{\cdots \otimes \mathsf{I}}_{i\text{ factors}}.
\end{equation}

([Equation (3) in the tutorial](https://www.stylewarning.com/posts/quantum-interpreter/))

$\otimes$ is the Kronecker-Product of two matrices:

\begin{aligned}
\begin{pmatrix}
a_{11} & a_{12} \\
a_{21} & a_{22}
\end{pmatrix}
\otimes
\begin{pmatrix}
b_{11} & b_{12} \\
b_{21} & b_{22}
\end{pmatrix}
&=
\begin{pmatrix}
a_{11} \begin{pmatrix}
b_{11} & b_{12} \\
b_{21} & b_{22}
\end{pmatrix} &
a_{12} \begin{pmatrix}
b_{11} & b_{12} \\
b_{21} & b_{22}
\end{pmatrix}\\
a_{21} \begin{pmatrix}
b_{11} & b_{12} \\
b_{21} & b_{22}
\end{pmatrix} &
a_{22} \begin{pmatrix}
b_{11} & b_{12} \\
b_{21} & b_{22}
\end{pmatrix}
\end{pmatrix} \\
&=
\begin{pmatrix}
a_{11} b_{11} &
a_{11} b_{12} &
a_{12} b_{11} &
a_{12} b_{12} \\
a_{11} b_{21} &
a_{11} b_{22} &
a_{12} b_{21} &
a_{12} b_{22} \\
a_{21} b_{11} &
a_{21} b_{12} &
a_{22} b_{11} &
a_{22} b_{12} \\
a_{21} b_{21} &
a_{21} b_{22} &
a_{22} b_{21} &
a_{22} b_{22}
\end{pmatrix}
\end{aligned}

In [524]:
from functools import reduce

def kroneckerExp(u: Operator, n: int) -> Operator:
    """Apply operator u n-times to itself with the Kronecker-product.
    
    Arguments:
        u       The operator for which to compute the Kronecker-exponential.
        n       The exponent of the Kronecker-exponential.
    """
    name = f"{u.name}^{n}"
    if n < 1:
        return Operator(name, np.ones((1,)))
    return Operator(name, reduce(np.kron, [u.matrix] * (n-1), u.matrix))

In [525]:
kroneckerExp(I, 1)

Operator I^1:
[[1 0]
 [0 1]]

In [526]:
kroneckerExp(I, 2)

Operator I^2:
[[1 0 0 0]
 [0 1 0 0]
 [0 0 1 0]
 [0 0 0 1]]

In [527]:
kroneckerExp(I, 3)

Operator I^3:
[[1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 1]]

In [528]:
def lift(u: Operator, i: int, n: int) -> Operator:
    """Lift a quantum gate to qubit i in a quantum state of n qubits.
    
    Arguments:
        u       The quantum gate to lift.
        i       The qubit to which the gate applies.
        n       The number of qubits in the quantum state.
    """
    left = kroneckerExp(I, n - i - u.numberOfQubits)
    right = kroneckerExp(I, i)
    return Operator(f"{u.name} on qubit {i} of {n}",
                    np.kron(left.matrix, np.kron(u.matrix, right.matrix)))

def applyOperatorToQubit(state: State, u: Operator, qubit: int):
    """Apply operator to qubit in state.
    
    Arguments:
        state       The quantum state.
        u           The operator to apply to the qubit in the state.
        qubit       The qubit to which the operator applies.
    """
    operator = lift(u, qubit, state.numberOfQubits)
    state.apply(operator)

In [529]:
# Apply X to qubit 0 of a 4-qubit machine, that is, swap |...0> and |...1>.
lift(X, 0, 4)

Operator X on qubit 0 of 4:
[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]]

In [530]:
state = State(4)
state.state = np.stack([1.0, 0.0] * 2 ** (state.numberOfQubits-1))
state

State of 4 qubits with amplitudes and probabilities:
|0000>                 1.0                 1.0
|0001>                 0.0                 0.0
|0010>                 1.0                 1.0
|0011>                 0.0                 0.0
|0100>                 1.0                 1.0
|0101>                 0.0                 0.0
|0110>                 1.0                 1.0
|0111>                 0.0                 0.0
|1000>                 1.0                 1.0
|1001>                 0.0                 0.0
|1010>                 1.0                 1.0
|1011>                 0.0                 0.0
|1100>                 1.0                 1.0
|1101>                 0.0                 0.0
|1110>                 1.0                 1.0
|1111>                 0.0                 0.0

In [531]:
applyOperatorToQubit(state, X, 0)
state

State of 4 qubits with amplitudes and probabilities:
|0000>                 0.0                 0.0
|0001>                 1.0                 1.0
|0010>                 0.0                 0.0
|0011>                 1.0                 1.0
|0100>                 0.0                 0.0
|0101>                 1.0                 1.0
|0110>                 0.0                 0.0
|0111>                 1.0                 1.0
|1000>                 0.0                 0.0
|1001>                 1.0                 1.0
|1010>                 0.0                 0.0
|1011>                 1.0                 1.0
|1100>                 0.0                 0.0
|1101>                 1.0                 1.0
|1110>                 0.0                 0.0
|1111>                 1.0                 1.0

### Gates on multiple non-adjacent qubits

We can apply gates to a quantum state if they are compatible in the number of qubits.

In [532]:
kroneckerExp(I, 3).shape[0]

AttributeError: 'Operator' object has no attribute 'shape'

In [ ]:
def applyOperator(
        state: State,
        u: Matrix,
        qubits: QubitList):
    """Apply operator u to qubits in state.
    
    Arguments:
        state       The quantum state.
        u           The operator to apply to the qubits in the state.
        qubits      A list of qubits to which the operator applies.
    """
    assert(len(qubits) == state.numberOfQubits)
    assert(len(qubits) == len(operator))

    match qubits:
        case [qubit]:
            applyOperatorToQubit(state, operator, qubit)
        case qubits:
            applyOperatorToQubits(state, operator, qubits)

In [ ]:
list(range(0, 4))

[0, 1, 2, 3]

In [ ]:
uc = np.unique_counts([0, 1, 2, 0, 1, 2, 0, 2, 2, 1, 1, 1, 1])
uc.values[np.argmax(uc.counts)]

np.int64(1)

In [ ]:
np.kron(I, X)

array([[0, 1, 0, 0],
       [1, 0, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 1, 0]])